In [ ]:
import sys
import os
from pathlib import Path

current_dir = str(Path(os.getcwd()).parent.parent.parent)
print(f"Current dir: {current_dir}")
sys.path.insert(0, current_dir)


In [ ]:
from external_modules.training_hyperparameters.hyperparameter_optimizer import (
    load_training_data,
    reset_hyperparameters,
    evaluate_base_scores,
    hpo_cycle,
    get_hpo_state,
)
from shared.config import config

# Load options from the single source of truth: training_config.json
training_cfg = config["training"]

# Read-only values pulled from training config
NUM_STEPS = int(training_cfg["iterations_per_cycle"])
NUM_CYCLES = int(training_cfg["cycles"])
MAX_COMBOS = int(training_cfg["max_combos"])
# ───────────────────────────────────────────────────────────────

# Load training data (no automatic retrain from config)
X, y = load_training_data(retrain=False)
print(f"Training data ready: X={X.shape}, y={y.shape}")


In [ ]:
# Guarded creation of base configs (top1..top4)
RESET = False  # set True to initialize base configs and persist to training_config.json
if RESET:
    # Force create base configs and persist to disk
    reset_hyperparameters(evaluate=False, X=X, y=y, force=True)
    print("Base configs created: top1..top4 written to training_config.json")
else:
    print(
        "RESET is False: skipping creation of base configs. To create them, set RESET=True and re-run this cell."
    )

# Run one HPO cycle (will raise if base configs are missing)
for cycle in range(NUM_CYCLES):
    print(f"\n=== Starting HPO Cycle {cycle + 1}/{NUM_CYCLES} ===")
    result = hpo_cycle(X, y, num_steps=NUM_STEPS, max_combos=MAX_COMBOS)
    print("HPO result summary:", result)
